In [0]:
%sql

CREATE OR REPLACE TABLE gold.fact_sales
USING DELTA
AS

SELECT

    s.order_number,

    s.customer_key,

    s.product_key,

    s.order_date,

    s.shipping_date,

    s.due_date,

    s.days_to_ship,

    s.quantity,

    s.price,

    s.sales_amount,

    p.cost,

    (s.quantity * p.cost) AS total_cost,

    (s.sales_amount - (s.quantity * p.cost)) AS profit,

    ROUND(

        ((s.sales_amount - (s.quantity * p.cost))
        / NULLIF(s.sales_amount,0))*100,

        2

    ) AS profit_margin,

    CASE

        WHEN s.days_to_ship <= 3 THEN 'Fast'

        WHEN s.days_to_ship <= 7 THEN 'Normal'

        ELSE 'Delayed'

    END AS shipping_performance,

    s.order_year,

    s.order_month

FROM silver.sales s

LEFT JOIN silver.products p

ON s.product_key = p.product_key;